# Notebook 2 · Heat islands, Local Climate Zones and walkability link
**Analysis code for Al-Homoud, M. "Shaded but Not Cooler" (F1000Research).** Run of 5 November 2026.
This notebook reads the outputs of Notebook 1 from Google Drive (`thermal_oldcore_alwaha`) and completes the analysis without re-running the 1 m model:
- **A.** Landsat 8/9 summer surface temperature and Getis–Ord Gi* hot and cold spots (Table 7, Figure 9).
- **B.** MODIS Aqua day and night surface urban heat island, 2013–2025 (Table 8, Figure 10).
- **C.** Local Climate Zones with the ESA WorldCover 10 m land-cover map (Figure 5).
- **D.** Link to the old-core walkability hexagons of the companion GIS study (Table 9, Figure 11; needs `oldcore_ks_out.zip`).
- **E.** One archive of all results, saved to Google Drive.

**How to run:** upload `oldcore_ks_out.zip` with the folder icon (left), then **Runtime → Run all** (about 20–40 minutes). Allow Google Drive access.

The satellite data come from the Microsoft Planetary Computer; if it is temporarily unavailable, the cells wait and retry automatically, and finished parts are kept.

In [ ]:
!pip -q install rasterio pystac-client planetary-computer geopandas pyogrio statsmodels

In [ ]:
%%writefile tw_extra.py
# ---- Extra analyses for the old-core vs Al-Waha study ----
# 1. Pedestrian realm from MEASURED street widths (works for narrow traditional lanes and wide villa streets)
# 2. Local Climate Zones (Stewart & Oke 2012) from the 1 m morphology, on a 100 m grid
# 3. Getis-Ord Gi* hot/cold spots on a raster, with Benjamini-Hochberg FDR
import math
import numpy as np
import pandas as pd


# ------------------------------------------------------------------ 1. pedestrian realm
def measured_half_width(e, free_dist, transform, res, step=2.0):
    """Median distance (m) from each segment centreline to the nearest building facade.
    free_dist: distance-to-nearest-building raster (m)."""
    T = transform
    H, W = free_dist.shape
    out = []
    for gm in e.geometry:
        n = max(2, int(gm.length // step))
        pts = [gm.interpolate(i / (n - 1), normalized=True) for i in range(n)]
        vals = []
        for p in pts:
            c, r = int((p.x - T.c) / T.a), int((p.y - T.f) / T.e)
            if 0 <= r < H and 0 <= c < W:
                vals.append(free_dist[r, c])
        v = np.array(vals, 'float32')
        v = v[v > 0]
        out.append(float(np.median(v)) if v.size else np.nan)
    return np.array(out)


def realm_geometry(gm, cls, half_class, hw_meas, sw_w=2.5, shared_max_w=8.0, plaza_half=4.0):
    """Return (pedestrian_polygon, carriageway_polygon_or_None, street_width_m, realm_type).
    Rules (applied identically to both districts):
      - pedestrian-only ways (footway/pedestrian/path/steps) -> a band of min(measured half-width, 4 m) each side;
      - lanes whose facade-to-facade width < shared_max_w (8 m) -> shared lane: the whole lane is pedestrian realm;
      - streets with room for a sidewalk -> 2.5 m sidewalk strips outside the carriageway, where the carriageway
        half-width is the class value, reduced when the facades are closer than class value + sidewalk."""
    if not np.isfinite(hw_meas) or hw_meas <= 0:
        hw_meas = half_class + sw_w                 # no buildings nearby: fall back to the class table
    W = 2 * hw_meas
    if half_class == 0:
        hw = min(hw_meas, plaza_half)
        return gm.buffer(max(hw, 1.0), cap_style=2), None, W, 'pedestrian way'
    if W < shared_max_w:
        return gm.buffer(max(hw_meas, 1.0), cap_style=2), None, W, 'shared lane'
    carr = min(half_class, max(hw_meas - sw_w, 1.5))
    ped = gm.buffer(carr + sw_w, cap_style=2).difference(gm.buffer(carr, cap_style=2))
    return ped, gm.buffer(carr, cap_style=2), W, 'street with sidewalks'


# ------------------------------------------------------------------ 2. Local Climate Zones
# Stewart & Oke (2012), Table 3: SVF, aspect ratio, building surface fraction (%), impervious surface fraction (%),
# pervious surface fraction (%), height of roughness elements (m). Only zones that can occur here are listed.
LCZ_TABLE = {
    'LCZ 1 compact high-rise': dict(svf=(0.2, 0.4), ar=(2, 6), bsf=(40, 60), isf=(40, 60), psf=(0, 10), h=(25, 80)),
    'LCZ 2 compact mid-rise':  dict(svf=(0.3, 0.6), ar=(0.75, 2), bsf=(40, 70), isf=(30, 50), psf=(0, 20), h=(10, 25)),
    'LCZ 3 compact low-rise':  dict(svf=(0.2, 0.6), ar=(0.75, 1.5), bsf=(40, 70), isf=(20, 50), psf=(0, 30), h=(3, 10)),
    'LCZ 4 open high-rise':    dict(svf=(0.5, 0.7), ar=(0.75, 1.25), bsf=(20, 40), isf=(30, 40), psf=(30, 40), h=(25, 80)),
    'LCZ 5 open mid-rise':     dict(svf=(0.5, 0.8), ar=(0.3, 0.75), bsf=(20, 40), isf=(30, 50), psf=(20, 40), h=(10, 25)),
    'LCZ 6 open low-rise':     dict(svf=(0.6, 0.9), ar=(0.3, 0.75), bsf=(20, 40), isf=(20, 50), psf=(30, 60), h=(3, 10)),
    'LCZ 8 large low-rise':    dict(svf=(0.7, 1.0), ar=(0.1, 0.3), bsf=(30, 50), isf=(40, 50), psf=(0, 20), h=(3, 10)),
    'LCZ 9 sparsely built':    dict(svf=(0.8, 1.0), ar=(0.1, 0.25), bsf=(10, 20), isf=(0, 20), psf=(60, 80), h=(3, 10)),
    'LCZ E bare rock or paved': dict(svf=(0.9, 1.0), ar=(0, 0.1), bsf=(0, 10), isf=(90, 100), psf=(0, 10), h=(0, 0.25)),
    'LCZ F bare soil or sand': dict(svf=(0.9, 1.0), ar=(0, 0.1), bsf=(0, 10), isf=(0, 10), psf=(90, 100), h=(0, 0.25)),
}
# Scales put the weight on 3D form (SVF, aspect ratio, building fraction, height): in desert cities paved and bare
# ground are hard to separate from open data, so surface-cover fractions get half the weight.
_SCALE = dict(svf=0.15, ar=0.3, bsf=12, isf=40, psf=40, h=5)


def lcz_classify(props, footprint=None, large_m2=1000.0):
    """props: dict of arrays (svf, ar, bsf, isf, psf, h). Nearest Stewart & Oke class by summed range distance.
    footprint: mean building footprint (m2) per cell. LCZ 8 is by definition large low-rise buildings
    (warehouses, malls), so it is only allowed where the mean footprint exceeds large_m2."""
    names = list(LCZ_TABLE)
    cost = []
    for n in names:
        c = 0
        for k, (lo, hi) in LCZ_TABLE[n].items():
            x = props[k]
            c = c + (np.clip(lo - x, 0, None) + np.clip(x - hi, 0, None)) / _SCALE[k]
        if n.startswith('LCZ 8') and footprint is not None:
            c = np.where(footprint >= large_m2, c, np.inf)
        cost.append(c)
    cost = np.stack(cost)
    idx = np.nanargmin(np.where(np.isfinite(cost), cost, np.inf), axis=0)
    return np.array(names, dtype=object)[idx], np.nanmin(cost, axis=0)


def morphology_cells(bh, can, svf, impervious, ground, grid, cell=100.0, fp_px=None):
    """Aggregate 1 m morphology to `cell` m cells. impervious: bool raster of paved ground (not buildings).
    fp_px: raster holding, at each building pixel, the footprint area (m2) of that building (from the polygons).
    Returns DataFrame with one row per cell (row, col, x, y, bsf, isf, psf, h, svf, ar, canopy) + LCZ."""
    T, W, H = grid
    res = T.a
    f = int(round(cell / res))
    h2, w2 = (H // f) * f, (W // f) * f
    def agg(a, how='mean'):
        a = a[:h2, :w2].reshape(h2 // f, f, w2 // f, f)
        return np.nanmean(a, axis=(1, 3)) if how == 'mean' else np.nansum(a, axis=(1, 3))
    b = (bh > 0).astype('float32')
    bsf = 100 * agg(b)
    hb = np.where(bh > 0, bh, np.nan)
    with np.errstate(invalid='ignore'):
        h = agg(np.nan_to_num(hb) * b, 'sum') / np.maximum(agg(b, 'sum'), 1)
    isf = 100 * agg((impervious & ground).astype('float32'))
    psf = np.clip(100 - bsf - isf, 0, 100)
    sv = agg(np.where(ground, svf, np.nan))
    canopy = 100 * agg((can > 0).astype('float32'))
    # aspect ratio H/W from the canyon relation SVF = cos(atan(2 H/W)) (Oke 1987), using the cell's mean ground SVF
    ar = np.clip(np.tan(np.arccos(np.clip(sv, 0.05, 1))) / 2, 0, 6)
    if fp_px is None:                                                   # fallback: connected building patches
        from scipy.ndimage import label, binary_erosion
        lab, _ = label(binary_erosion(bh > 0))
        size = np.bincount(lab.ravel()) * res * res
        fp_px = np.where(lab > 0, size[lab], np.nan).astype('float32')
    fp_px = np.where(bh > 0, fp_px, np.nan)
    footprint = agg(fp_px)                                              # area-weighted mean footprint per cell
    lcz, cost = lcz_classify(dict(svf=sv, ar=ar, bsf=bsf, isf=isf, psf=psf, h=h), footprint)
    rr, cc = np.mgrid[0:h2 // f, 0:w2 // f]
    x = T.c + (cc + 0.5) * cell
    y = T.f - (rr + 0.5) * cell
    return pd.DataFrame({'row': rr.ravel(), 'col': cc.ravel(), 'x': x.ravel(), 'y': y.ravel(), 'bsf': bsf.ravel(),
                         'isf': isf.ravel(), 'psf': psf.ravel(), 'h': h.ravel(), 'svf': sv.ravel(), 'ar': ar.ravel(),
                         'canopy_pct': canopy.ravel(), 'footprint_m2': footprint.ravel(), 'lcz': lcz.ravel(),
                         'lcz_fit_cost': cost.ravel()})


# ------------------------------------------------------------------ 3. Getis-Ord Gi* on a raster
def getis_ord_star(a, radius_px):
    """Gi* z-scores with binary weights inside a circle of radius_px (self included). NaN cells are ignored."""
    from scipy.ndimage import convolve
    from scipy.stats import norm
    ok = np.isfinite(a)
    x = np.where(ok, a, 0.0).astype('float64')
    n = ok.sum()
    xbar = x[ok].mean()
    s = np.sqrt((x[ok] ** 2).mean() - xbar ** 2)
    r = int(radius_px)
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    k = ((xx ** 2 + yy ** 2) <= r * r).astype('float64')
    sw = convolve(ok.astype('float64'), k, mode='constant')          # sum of weights (valid neighbours)
    swx = convolve(x, k, mode='constant')
    with np.errstate(invalid='ignore', divide='ignore'):
        z = (swx - xbar * sw) / (s * np.sqrt((n * sw - sw ** 2) / (n - 1)))
    z[~ok] = np.nan
    p = 2 * norm.sf(np.abs(np.nan_to_num(z)))
    p[~ok] = np.nan
    return z.astype('float32'), p.astype('float32')


def bh_fdr(p, alpha=0.05):
    """Benjamini-Hochberg: boolean array, True where significant at FDR alpha."""
    pv = p[np.isfinite(p)]
    if pv.size == 0:
        return np.zeros(p.shape, bool)
    srt = np.sort(pv)
    m = srt.size
    below = srt <= alpha * np.arange(1, m + 1) / m
    thr = srt[np.nonzero(below)[0].max()] if below.any() else -1
    return np.isfinite(p) & (p <= thr)

In [ ]:
import os, sys, glob, math, time, json, zipfile, subprocess, warnings, shutil
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, rasterio
from shapely.geometry import box
from rasterio.features import rasterize
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_origin
from scipy.stats import spearmanr
warnings.filterwarnings('ignore'); sys.path.insert(0, '.')
import importlib, tw_extra; importlib.reload(tw_extra)
from tw_extra import getis_ord_star, bh_fdr, morphology_cells

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/thermal_oldcore_alwaha'
assert os.path.isdir(OUT), 'Folder thermal_oldcore_alwaha not found in My Drive - was the main notebook run with the same Google account?'
FIG = f'{OUT}/figures'; os.makedirs(FIG, exist_ok=True)
CRS_M = 'EPSG:32638'
LST_YEARS, LST_MONTHS, LST_MAX_CLOUD = (2023, 2025), (6, 9), 10
CONTEXT_M, GI_RADIUS_M, LCZ_CELL = 1500, 120, 100
MODIS_YEARS, MODIS_MONTHS = (2013, 2025), (6, 9)
CITY_CENTRE, RURAL_RING_KM = (24.6877, 46.7219), (40, 70)

# ---- what the main run left in Drive
D = {}
for name, folder in [('Old core', 'Old_core'), ('Al-Waha', 'Al-Waha')]:
    d = {'dir': f'{OUT}/{folder}'}
    d['poly_m'] = gpd.read_file(f"{d['dir']}/boundary.geojson").to_crs(CRS_M).geometry.iloc[0]
    with rasterio.open(f"{d['dir']}/building_height_1m.tif") as s:
        d['bh'] = s.read(1).astype('float32'); d['grid'] = (s.transform, s.width, s.height)
    d['bh'][d['bh'] < 0] = 0
    D[name] = d
if os.path.exists(f'{OUT}/design_day_weather.csv'):
    DAY = str(pd.read_csv(f'{OUT}/design_day_weather.csv', parse_dates=[0]).iloc[0, 0].date())
else:
    f = sorted(glob.glob(f"{D['Old core']['dir']}/utci_ge46_hours_*.tif")); DAY = os.path.basename(f[0])[16:26] if f else 'unknown'
print('Design day:', DAY)
for f in ['segments_results.csv', 'districts_hourly.csv', 'table_scenarios.csv', 'table_solweig_crosscheck.csv', 'table_district_comparison.csv',
          'table_models.csv', 'lcz_cells.csv']:
    print(('OK      ' if os.path.exists(f'{OUT}/{f}') else 'MISSING ') + f)
for n, d in D.items():
    print(n, '| key-hour rasters found:', len(glob.glob(f"{d['dir']}/*_1500_*.tif")), '| daily rasters:', len(glob.glob(f"{d['dir']}/utci_ge46_hours_*.tif")))

In [ ]:
# ---- helpers: patient Planetary Computer access (it rate-limits shared Colab machines), open AWS mirrors
import pystac_client, planetary_computer
PC_URL = 'https://planetarycomputer.microsoft.com/api/stac/v1'
def pc_search(**kw):
    for k in range(8):
        try:
            cli = pystac_client.Client.open(PC_URL, modifier=planetary_computer.sign_inplace)
            return list(cli.search(**kw).items())
        except Exception as e:
            w = 45 * (k + 1); print(f'   Planetary Computer busy ({str(e)[:60]}) - waiting {w} s'); time.sleep(w)
    raise RuntimeError('Planetary Computer still busy - run this cell again in about an hour')

def make_grid(poly_m, res, buf):
    x0, y0, x1, y1 = poly_m.buffer(buf).bounds
    x0, y0 = math.floor(x0 / 30) * 30, math.floor(y0 / 30) * 30
    x1, y1 = math.ceil(x1 / 30) * 30, math.ceil(y1 / 30) * 30
    return from_origin(x0, y1, res, res), int(round((x1 - x0) / res)), int(round((y1 - y0) / res))

def read_to_grid(path, grid, resampling=Resampling.bilinear, ovr=None, tries=4):
    T, W, H = grid
    for k in range(tries):
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', AWS_NO_SIGN_REQUEST='YES', GDAL_HTTP_MAX_RETRY='5', GDAL_HTTP_RETRY_DELAY='3'):
                try: src = rasterio.open(path, overview_level=ovr) if ovr is not None else rasterio.open(path)
                except Exception: src = rasterio.open(path)
                with src, WarpedVRT(src, crs=CRS_M, transform=T, width=W, height=H, resampling=resampling) as vrt:
                    return vrt.read(1, masked=True).astype('float32').filled(np.nan)
        except Exception as e:
            if k == tries - 1: raise
            time.sleep(10 * (k + 1))

def save_tif(arr, path, grid, nodata=-9999):
    T, W, H = grid
    with rasterio.open(path, 'w', driver='GTiff', height=H, width=W, count=1, dtype='float32', crs=CRS_M, transform=T, nodata=nodata, compress='deflate') as d:
        d.write(np.where(np.isfinite(arr), arr, nodata).astype('float32'), 1)

def inside_mask(poly_m, grid):
    T, W, H = grid
    return rasterize([(poly_m, 1)], out_shape=(H, W), transform=T, fill=0, dtype='uint8').astype(bool)

def to_gpkg(gdf, path):
    tmp = '/content/_tmp_' + os.path.basename(path)
    if os.path.exists(tmp): os.remove(tmp)
    gdf.to_file(tmp, driver='GPKG'); shutil.copy(tmp, path)

def extent(grid):
    T, W, H = grid
    return [T.c, T.c + W * T.a, T.f + H * T.e, T.f]

def worldcover_aws(grid, ovr=None):
    # ESA WorldCover 2021 v200 from its public AWS bucket (no account, no rate limit)
    out = np.full((grid[2], grid[1]), np.nan, 'float32')
    for t in ['N24E045', 'N21E045', 'N24E048', 'N21E042', 'N24E042']:
        url = f'/vsicurl/https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_{t}_Map.tif'
        try: out = np.where(np.isnan(out), read_to_grid(url, grid, Resampling.mode if ovr is not None else Resampling.nearest, ovr=ovr, tries=2), out)
        except Exception: pass
    return out

def cop_dem_aws(grid, ovr=None):
    out = np.full((grid[2], grid[1]), np.nan, 'float32')
    for lat in (23, 24, 25):
        for lon in (45, 46, 47):
            t = f'Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM'
            try: out = np.where(np.isnan(out), read_to_grid(f'/vsicurl/https://copernicus-dem-30m.s3.amazonaws.com/{t}/{t}.tif', grid, Resampling.average, ovr=ovr, tries=2), out)
            except Exception: pass
    return out
print('helpers ready')

## A · Landsat summer surface temperature, context contrast and hot/cold spots (≈5–15 min)

In [ ]:
def gi_classes(z, s99, s95, s90):
    c = np.zeros(z.shape, 'int8')
    for s, v in ((s90, 1), (s95, 2), (s99, 3)): c = np.where(s & (z > 0), v, np.where(s & (z < 0), -v, c))
    return c

LST_ROWS = []
for name, d in D.items():
    g30 = make_grid(d['poly_m'], 30, CONTEXT_M)
    bbox = list(gpd.GeoSeries([d['poly_m'].buffer(CONTEXT_M)], crs=CRS_M).to_crs(4326).total_bounds)
    items = pc_search(collections=['landsat-c2-l2'], bbox=bbox, datetime=f'{LST_YEARS[0]}-01-01/{LST_YEARS[1]}-12-31',
                      query={'eo:cloud_cover': {'lt': LST_MAX_CLOUD}, 'platform': {'in': ['landsat-8', 'landsat-9']}})
    items = [i for i in items if LST_MONTHS[0] <= i.datetime.month <= LST_MONTHS[1]]
    print(f'{name}: {len(items)} summer scenes found')
    stack = []
    for i, it in enumerate(items):
        if i and i % 25 == 0:                          # signed links expire: re-sign the rest
            items[i:] = [planetary_computer.sign(x) for x in items[i:]]
        try:
            st = read_to_grid(it.assets['lwir11'].href, g30); qa = read_to_grid(it.assets['qa_pixel'].href, g30, Resampling.nearest)
        except Exception as e:
            print('   skipped', it.id, str(e)[:50]); continue
        q = np.nan_to_num(qa, nan=1).astype('int64'); lst = st * 0.00341802 + 149.0 - 273.15
        lst[~np.isfinite(st) | (st <= 0) | ((q & (1 | 2 | 8 | 16)) > 0)] = np.nan
        if np.mean(np.isfinite(lst)) > 0.5: stack.append(lst)
        time.sleep(0.3)
    lst = np.nanmedian(np.stack(stack), 0).astype('float32')
    save_tif(lst, f"{d['dir']}/LST_summer_median_30m_context.tif", g30)
    ins = inside_mask(d['poly_m'], g30)
    ring = inside_mask(d['poly_m'].buffer(CONTEXT_M), g30) & ~inside_mask(d['poly_m'].buffer(60), g30)
    z, p = getis_ord_star(lst, GI_RADIUS_M / 30)
    cls = gi_classes(z, bh_fdr(p, .01), bh_fdr(p, .05), bh_fdr(p, .10))
    save_tif(z, f"{d['dir']}/LST_GiStar_z_30m.tif", g30); save_tif(cls.astype('float32'), f"{d['dir']}/LST_hotspot_class_30m.tif", g30)
    d.update(lst30=lst, g30=g30, gi=cls)
    row = {'district': name, 'scenes': len(stack), 'LST_district_mean': float(np.nanmean(lst[ins])), 'LST_ring_mean': float(np.nanmean(lst[ring])),
           'contrast_district_minus_ring': float(np.nanmean(lst[ins]) - np.nanmean(lst[ring])),
           'hot_spot_99_pct_area': 100 * float(np.mean(cls[ins] == 3)), 'hot_spot_any_pct_area': 100 * float(np.mean(cls[ins] > 0)),
           'cold_spot_99_pct_area': 100 * float(np.mean(cls[ins] == -3)), 'cold_spot_any_pct_area': 100 * float(np.mean(cls[ins] < 0))}
    LST_ROWS.append(row); print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
LST_TABLE = pd.DataFrame(LST_ROWS); LST_TABLE.to_csv(f'{OUT}/table_landsat_lst_hotspots.csv', index=False)

# LST on every street segment (mean along the street line, every 10 m) and its link with pedestrian heat
SEG = gpd.read_file(f'{OUT}/segments_results.gpkg').to_crs(CRS_M) if os.path.exists(f'{OUT}/segments_results.gpkg') else None
if SEG is None: print('segments_results.gpkg not in Drive - street-level LST skipped')
def lst_along(gm, d):
    T = d['g30'][0]; H, W = d['lst30'].shape; v = []
    for k in range(max(2, int(gm.length // 10))):
        p = gm.interpolate(k / max(1, int(gm.length // 10) - 1), normalized=True)
        r, c = int((T.f - p.y) / 30), int((p.x - T.c) / 30)
        if 0 <= r < H and 0 <= c < W: v.append(d['lst30'][r, c])
    return float(np.nanmean(v)) if v else np.nan
if SEG is not None:
    SEG['lst'] = [lst_along(gm, D[n]) for gm, n in zip(SEG.geometry, SEG['district'])]
    to_gpkg(SEG, f'{OUT}/segments_results.gpkg'); SEG.drop(columns='geometry').to_csv(f'{OUT}/segments_results.csv', index=False)
    for n, g in SEG.groupby('district'):
        g = g.dropna(subset=['lst', 'utci_15'])
        print(f"{n}: Spearman ρ(LST, UTCI 15:00) = {spearmanr(g.lst, g.utci_15).statistic:.2f} (n = {len(g)})")

# figure
from matplotlib.colors import ListedColormap, BoundaryNorm
fig, axs = plt.subplots(2, 2, figsize=(11, 10))
vmin = min(np.nanpercentile(d['lst30'], 2) for d in D.values()); vmax = max(np.nanpercentile(d['lst30'], 98) for d in D.values())
gcm = ListedColormap(['#2166ac', '#67a9cf', '#d1e5f0', '#f7f7f7', '#fddbc7', '#ef8a62', '#b2182b']); gn = BoundaryNorm(np.arange(-3.5, 4.5), 7)
for r, (n, d) in enumerate(D.items()):
    ex = extent(d['g30'])
    im = axs[r, 0].imshow(d['lst30'], extent=ex, cmap='magma', vmin=vmin, vmax=vmax); im2 = axs[r, 1].imshow(d['gi'], extent=ex, cmap=gcm, norm=gn)
    for a in axs[r]: gpd.GeoSeries([d['poly_m']]).boundary.plot(ax=a, color='#00b7c7' if a is axs[r, 0] else 'k', lw=1.2); a.set_xticks([]); a.set_yticks([])
    axs[r, 0].set_title(f'{n}: summer LST (Landsat 8/9)'); axs[r, 1].set_title(f'{n}: hot and cold spots (Gi*)')
    plt.colorbar(im, ax=axs[r, 0], shrink=.7, label='°C')
    cb = plt.colorbar(im2, ax=axs[r, 1], shrink=.7, ticks=range(-3, 4)); cb.ax.set_yticklabels(['cold 99%', '95%', '90%', 'n.s.', '90%', '95%', 'hot 99%'])
fig.tight_layout(); fig.savefig(f'{FIG}/F5_lst_hotspots.png', dpi=300); plt.show()

## B · MODIS Aqua day (13:30) and night (01:30) heat island against the desert (≈10–25 min)

In [ ]:
def asset_key(item, *must):
    for k in item.assets:
        if all(m.lower() in k.lower() for m in must): return k
    return None

c = gpd.GeoSeries(gpd.points_from_xy([CITY_CENTRE[1]], [CITY_CENTRE[0]]), crs=4326).to_crs(CRS_M).iloc[0]
mg = (from_origin(math.floor((c.x - 80000) / 1000) * 1000, math.ceil((c.y + 80000) / 1000) * 1000, 1000, 1000), 160, 160)
bbox = list(gpd.GeoSeries([box(*gpd.GeoSeries([c], crs=CRS_M).buffer(80000).total_bounds)], crs=CRS_M).to_crs(4326).total_bounds)
MOD = {}
for y in range(MODIS_YEARS[0], MODIS_YEARS[1] + 1):
    items = pc_search(collections=['modis-11A2-061'], bbox=bbox, datetime=f'{y}-{MODIS_MONTHS[0]:02d}-01/{y}-{MODIS_MONTHS[1]:02d}-30')
    items = [i for i in items if i.id.upper().startswith('MYD') or str(i.properties.get('platform', '')).lower() == 'aqua']
    acc = {'day': [], 'night': []}
    for it in items:
        for part, key in (('day', asset_key(it, 'lst_day')), ('night', asset_key(it, 'lst_night'))):
            if key is None: continue
            try: a = read_to_grid(it.assets[key].href, mg, Resampling.nearest)
            except Exception: continue
            qk = asset_key(it, 'qc_' + part)
            if qk:
                try:
                    qc = np.nan_to_num(read_to_grid(it.assets[qk].href, mg, Resampling.nearest), nan=255).astype('int64')
                    a[((qc & 3) > 1) | (((qc >> 6) & 3) > 1)] = np.nan
                except Exception: pass
            acc[part].append(np.where(a > 7500, a * 0.02 - 273.15, np.nan)); time.sleep(0.2)
    MOD[y] = {k: (np.nanmean(np.stack(v), 0) if v else None) for k, v in acc.items()}
    print(f"{y}: {len(items)} Aqua composites | day {len(acc['day'])} | night {len(acc['night'])}")

# rural desert reference (land cover and elevation from open AWS mirrors)
T, W, H = mg
X, Y = np.meshgrid(T.c + (np.arange(W) + .5) * 1000, T.f - (np.arange(H) + .5) * 1000)
dist = np.hypot(X - c.x, Y - c.y) / 1000
RURAL = (dist >= RURAL_RING_KM[0]) & (dist <= RURAL_RING_KM[1]); note = ['distance']
wc1 = worldcover_aws(mg, ovr=4)
if np.isfinite(wc1).any():
    from scipy.ndimage import binary_dilation
    RURAL &= np.isin(wc1, [20, 30, 60]) & ~binary_dilation(wc1 == 50, iterations=3); note.append('land cover')
dem1 = cop_dem_aws(mg, ovr=3)
if np.isfinite(dem1).any():
    cx, cy = int((c.x - T.c) / 1000), int((T.f - c.y) / 1000)
    RURAL &= np.abs(dem1 - np.nanmedian(dem1[cy - 10:cy + 10, cx - 10:cx + 10])) <= 150; note.append('elevation')
print(f'Rural reference: {int(RURAL.sum())} pixels of 1 km² (filters: {" + ".join(note)})')
save_tif(RURAL.astype('float32'), f'{OUT}/MODIS_rural_reference_mask.tif', mg)

rows = []
fine = (from_origin(T.c, T.f, 100, 100), W * 10, H * 10)
for name, d in D.items():
    ins = inside_mask(d['poly_m'], fine)
    for y, v in MOD.items():
        row = {'district': name, 'year': y}
        for part in ('day', 'night'):
            if v[part] is None: continue
            row[f'LST_{part}'] = float(np.nanmean(np.repeat(np.repeat(v[part], 10, 0), 10, 1)[ins]))
            row[f'rural_{part}'] = float(np.nanmedian(v[part][RURAL])); row[f'SUHI_{part}'] = row[f'LST_{part}'] - row[f'rural_{part}']
        rows.append(row)
MODIS_TABLE = pd.DataFrame(rows); MODIS_TABLE.to_csv(f'{OUT}/table_modis_suhi_by_year.csv', index=False)
display(MODIS_TABLE.groupby('district')[[k for k in MODIS_TABLE if k.startswith(('LST', 'SUHI'))]].agg(['mean', 'std']).round(2))
MEAN = {}
for part in ('day', 'night'):
    st = [v[part] for v in MOD.values() if v[part] is not None]
    if st: MEAN[part] = np.nanmean(np.stack(st), 0); save_tif(MEAN[part], f'{OUT}/MODIS_summer_{part}_LST_mean_1km.tif', mg)

COL = {'Old core': '#9e2a1f', 'Al-Waha': '#2a6f97'}
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5)); ex = extent(mg)
for a, part in zip(axs[:2], ['day', 'night']):
    if part in MEAN:
        im = a.imshow(MEAN[part], extent=ex, cmap='inferno'); plt.colorbar(im, ax=a, shrink=.7, label='°C')
        a.contour(np.flipud(RURAL.astype(float)), levels=[.5], colors='cyan', linewidths=.6, extent=ex, origin='lower')
        for d in D.values(): gpd.GeoSeries([d['poly_m']]).boundary.plot(ax=a, color='white', lw=1.2)
        a.set_title(f'Summer LST {"13:30" if part == "day" else "01:30"} ({MODIS_YEARS[0]}–{MODIS_YEARS[1]})'); a.set_xticks([]); a.set_yticks([])
for n, g in MODIS_TABLE.groupby('district'):
    if 'SUHI_day' in g: axs[2].plot(g.year, g.SUHI_day, '-o', ms=3, color=COL[n], label=f'{n}, day')
    if 'SUHI_night' in g: axs[2].plot(g.year, g.SUHI_night, '--s', ms=3, color=COL[n], label=f'{n}, night')
axs[2].axhline(0, color='k', lw=.6); axs[2].set_ylabel('SUHI vs desert (°C)'); axs[2].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(f'{FIG}/F6_modis_day_night.png', dpi=300); plt.show()

## C · Local Climate Zones with the real land-cover map (≈3 min)

In [ ]:
def find(d, pat):
    f = sorted(glob.glob(f"{d['dir']}/{pat}")); return f[0] if f else None
LCZ = []
for name, d in D.items():
    g = d['grid']; T, W, H = g
    rd = lambda p: rasterio.open(p).read(1).astype('float32')
    pc_, ps_ = find(d, 'CDSM_relative_*m.tif'), find(d, 'SVF_*m.tif')
    if ps_ is None:
        print(f'{name}: SVF raster not in Drive - LCZ kept from the main run'); continue
    can = np.zeros((H, W), 'float32')
    if pc_:
        c_ = rd(pc_); can = np.where(np.isfinite(c_) & (c_ > 0), c_, 0).astype('float32')
    svf = rd(ps_); svf[svf < -1] = np.nan
    wc = worldcover_aws(g)
    if not np.isfinite(wc).any():
        print(f'{name}: WorldCover not read - LCZ kept from the main run'); continue
    save_tif(wc, f"{d['dir']}/landcover_worldcover_1m.tif", g)
    b = gpd.read_file(find(d, 'buildings_with_heights.gpkg')).to_crs(CRS_M)
    fp = rasterize(((gm, gm.area) for gm in b.geometry), out_shape=(H, W), transform=T, fill=0, dtype='float32')
    ground = d['bh'] == 0
    cells = morphology_cells(d['bh'], can, svf, wc == 50, ground, g, LCZ_CELL, fp_px=np.where(d['bh'] > 0, fp, np.nan))
    f = int(LCZ_CELL); H2, W2 = (H // f) * f, (W // f) * f
    agg = lambda a: np.nanmean(a[:H2, :W2].reshape(H2 // f, f, W2 // f, f), axis=(1, 3)).ravel()
    cells['inside'] = agg(inside_mask(d['poly_m'], g).astype('float32')) >= .5
    for col, pat in [('utci_15', 'utci_1500_*.tif'), ('shade_15', 'shade_1500_*.tif'), ('utci_ge46_hours', 'utci_ge46_hours_*.tif'), ('tmrt_mean', 'tmrt_mean_*.tif')]:
        p = find(d, pat)
        if p:
            a = rd(p); a[a < -1000] = np.nan; cells[col] = agg(np.where(ground, a, np.nan)) * (100 if col == 'shade_15' else 1)
    cells['district'] = name; cells = cells[cells.inside]; LCZ.append(cells)
    print(name); display(cells.lcz.value_counts(normalize=True).mul(100).round(1).rename('% of district').to_frame())
if LCZ:
    if os.path.exists(f'{OUT}/lcz_cells.csv'): shutil.copy(f'{OUT}/lcz_cells.csv', f'{OUT}/lcz_cells_mainrun_placeholder_landcover.csv')
    LCZ = pd.concat(LCZ, ignore_index=True); LCZ.to_csv(f'{OUT}/lcz_cells.csv', index=False)
    to_gpkg(gpd.GeoDataFrame(LCZ, geometry=[box(x - 50, y - 50, x + 50, y + 50) for x, y in zip(LCZ.x, LCZ.y)], crs=CRS_M), f'{OUT}/lcz_cells.gpkg')
    display(LCZ.groupby(['district', 'lcz'])[['bsf', 'h', 'svf', 'utci_15', 'utci_ge46_hours']].median().round(2))

## D · Link to your walkability hexagons (upload `oldcore_ks_out.zip` first)

In [ ]:
GIS_ZIP = '/content/oldcore_ks_out.zip'
cand = [GIS_ZIP] + glob.glob('/content/oldcore_ks_out*.zip') + glob.glob('/content/drive/MyDrive/**/oldcore_ks_out*.zip', recursive=True)
GIS_ZIP = next((p for p in cand if os.path.exists(p)), None)
if GIS_ZIP:
    import pyogrio
    zdir = '/content/gis_run'; zipfile.ZipFile(GIS_ZIP).extractall(zdir); print('using', GIS_ZIP)
    best = None
    for f in glob.glob(f'{zdir}/**/*.gpkg', recursive=True) + glob.glob(f'{zdir}/**/*.geojson', recursive=True):
        try: layers = [l[0] for l in pyogrio.list_layers(f)]
        except Exception: layers = [None]
        for l in layers:
            try: g = gpd.read_file(f, layer=l) if l else gpd.read_file(f)
            except Exception: continue
            if len(g) < 15 or not g.geom_type.isin(['Polygon', 'MultiPolygon']).all(): continue
            a = g.to_crs(CRS_M).area.median(); score = ('hex' in f'{f}{l}'.lower()) * 2 + (20000 < a < 120000) + len(g.columns) / 100
            if best is None or score > best[0]: best = (score, f, l, len(g))
    if best is None or best[0] < 1:
        print('No hexagon grid found in the zip. Files inside:', [os.path.relpath(x, zdir) for x in glob.glob(f'{zdir}/**/*.*', recursive=True)][:40])
    else:
        print('hexagon grid:', os.path.relpath(best[1], zdir), best[2], f'({best[3]} hexagons)')
        HEX = gpd.read_file(best[1], layer=best[2]).to_crs(CRS_M).reset_index(drop=True); HEX['hex_id'] = np.arange(len(HEX))
        rows = []
        for n, d in D.items():
            T, W, H = d['grid']; ground = d['bh'] == 0; ins = inside_mask(d['poly_m'], d['grid'])
            lab = rasterize(((gm, i + 1) for gm, i in zip(HEX.geometry, HEX.hex_id)), out_shape=(H, W), transform=T, fill=0, dtype='int32')
            lab[~(ground & ins)] = 0; nh = len(HEX); cnt = np.bincount(lab.ravel(), minlength=nh + 1)
            r = pd.DataFrame({'hex_id': HEX.hex_id, 'open_px': cnt[1:]})
            for col, pat in [('utci_15', 'utci_1500_*.tif'), ('utci_12', 'utci_1200_*.tif'), ('shade_15_pct', 'shade_1500_*.tif'),
                             ('utci_ge46_hours', 'utci_ge46_hours_*.tif'), ('tmrt_mean', 'tmrt_mean_*.tif')]:
                p = sorted(glob.glob(f"{d['dir']}/{pat}"))
                if not p: continue
                a = rasterio.open(p[0]).read(1).astype('float32'); a[a < -1000] = np.nan; ok = (lab > 0) & np.isfinite(a)
                s = np.bincount(lab[ok], weights=a[ok], minlength=nh + 1); k = np.bincount(lab[ok], minlength=nh + 1)
                with np.errstate(invalid='ignore', divide='ignore'): r[col] = (s / k)[1:] * (100 if 'shade' in col else 1)
            if 'lst30' in d:
                L = rasterize(((gm, i + 1) for gm, i in zip(HEX.geometry, HEX.hex_id)), out_shape=d['lst30'].shape, transform=d['g30'][0], fill=0, dtype='int32')
                ok = (L > 0) & np.isfinite(d['lst30'])
                with np.errstate(invalid='ignore', divide='ignore'):
                    r['lst'] = (np.bincount(L[ok], weights=d['lst30'][ok], minlength=nh + 1) / np.bincount(L[ok], minlength=nh + 1))[1:]
            r = r[r.open_px > 500]; r['thermal_district'] = n; rows.append(r)
        HEXT = HEX.merge(pd.concat(rows), on='hex_id', how='inner')
        to_gpkg(HEXT, f'{OUT}/hexagons_walkability_thermal.gpkg'); HEXT.drop(columns='geometry').to_csv(f'{OUT}/hexagons_walkability_thermal.csv', index=False)
        thermal = ['utci_15', 'utci_12', 'shade_15_pct', 'utci_ge46_hours', 'lst']
        wcols = [c for c in HEXT.columns if pd.api.types.is_numeric_dtype(HEXT[c]) and c not in thermal + ['hex_id', 'open_px', 'tmrt_mean']
                 and any(k in c.lower() for k in ('wi', 'walk', 'uf', 'kswi', 'z_', 'gi', 'score', 'index'))]
        out = []
        for c_ in wcols:
            for t in [x for x in thermal if x in HEXT]:
                ok = HEXT[[c_, t]].dropna()
                if len(ok) > 10 and ok[c_].nunique() > 3: out.append({'walkability_metric': c_, 'thermal_metric': t, 'rho': spearmanr(ok[c_], ok[t]).statistic, 'n': len(ok)})
        HEXCOR = pd.DataFrame(out); HEXCOR.to_csv(f'{OUT}/table_hex_walkability_vs_heat.csv', index=False)
        print(f'{len(HEXT)} hexagons linked | walkability columns used: {wcols}')
        if len(HEXCOR): display(HEXCOR.sort_values('rho').round(2))
else:
    print('oldcore_ks_out.zip not found - upload it with the folder icon on the left and run this cell again.')

## E · One results zip (Google Drive + download)

In [ ]:
zpath = '/content/oldcore_alwaha_thermal_results.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for f in files:
            if f.endswith(('.parquet', '.src')) or 'cache' in root: continue
            if f.endswith('.tif') and os.path.getsize(os.path.join(root, f)) > 60e6: continue
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUT))
print(f'{zpath}: {os.path.getsize(zpath)/1e6:.0f} MB')
shutil.copy(zpath, '/content/drive/MyDrive/oldcore_alwaha_thermal_results.zip')
print('Saved in Google Drive > My Drive > oldcore_alwaha_thermal_results.zip')
try:
    from google.colab import files; files.download(zpath)
except Exception:
    print('Browser download did not start - take the zip from Google Drive.')